# GPT 시리즈: 언어모델의 스케일링 - 실습 코드 2: OpenAI API로 GPT-4 Few-shot + RLHF 체험

- Tutorial ID: `expand-gpt-series`
- Tutorial: GPT 시리즈: 언어모델의 스케일링
- Section ID: `expand-gpt-series-code-2`
- Section: 실습 코드 2: OpenAI API로 GPT-4 Few-shot + RLHF 체험

---

이번 실습에서는 OpenAI의 `gpt-4` 모델을 API로 직접 호출하면서, "완성된 GPT-4를 실제로 쓸 때 관찰할 수 있는 세 가지 특징"을 확인합니다.

- 🧩 **Few-shot Learning**: 예시 몇 개만으로 새로운 작업을 수행하는 능력
- 🛡️ **RLHF의 효과**: 안전하고 맥락에 맞게 답하도록 조정된 태도
- 🔧 **Function Calling**: 필요할 때 외부 함수(도구)를 스스로 골라 호출하는 능력

코드를 실행하기 전에 먼저 각 개념을 짧게 설명한 뒤, 바로 이어서 실습 코드와 결과 해석을 붙여두었습니다. 개념 설명 없이 코드부터 등장하는 부분은 없도록 구성했으니, 순서대로 천천히 따라가면 됩니다.


## 학습 목표

이전 실습(실습 코드 1)이 Transformer 내부의 Q/K/V, attention score 같은 **행렬 연산 레벨**을 직접 들여다보는 것이었다면, 이번 실습은 그 결과물인 "완성된 GPT-4 모델을 API로 사용할 때 어떤 현상이 벌어지는가"를 관찰합니다. 내부 수식을 보는 대신, **입력(prompt)을 어떻게 구성하느냐에 따라 출력이 어떻게 달라지는지**를 눈으로 확인하는 것이 이번 실습의 핵심입니다.

이 노트북을 다 풀고 나면 아래 세 가지를 스스로 설명할 수 있게 됩니다.

1. **Few-shot Learning (퓨샷 러닝)**
   모델 가중치를 전혀 업데이트하지 않고, prompt 안에 예시 몇 개만 넣어주는 것만으로 모델이 패턴을 파악해 새로운 입력에 적용하는 현상.
2. **RLHF (Reinforcement Learning from Human Feedback, 인간 피드백 기반 강화학습)의 효과**
   사전학습만 거친 모델과 달리, 민감하거나 맥락이 불분명한 요청에는 조심스럽게 반응하고 정당한 요청에는 적극적으로 돕는 "정렬(alignment)"된 태도.
3. **Function Calling (함수 호출)**
   모델이 직접 함수를 실행하는 게 아니라, "이 함수를 이 인자로 호출해줘"라는 제안을 만들고, 그 결과를 다시 받아 최종 답변을 완성하는 전체 흐름.

## 이 노트북을 읽는 순서

1. **0. 사전 준비**: 패키지 설치와 API 키 등록. (이 단계를 건너뛰면 이후 모든 코드에서 오류가 납니다.)
2. **1 → 2 → 3장**: `개념 설명(markdown) → 실습 코드 → 결과 해석`의 3단 구성이 3번 반복됩니다 (Few-shot / RLHF / Function Calling 순서).
3. 모든 코드 셀은 **줄 단위 주석**으로 "이 줄이 왜 필요한지"를 설명해두었습니다. 결과만 보지 말고 주석과 함께 읽어주세요.
4. 마지막 **정리** 섹션에서 세 개념을 표로 비교하며 복습하고, 직접 실험해볼 수 있는 선택 과제도 제공합니다.

## 주의할 점

- 이 노트북은 **유료 API 호출**을 포함합니다. 처음에는 셀을 하나씩 순서대로만 실행해서 비용을 최소화하고, 익숙해진 뒤에 자유롭게 프롬프트를 바꿔 실험해보세요.
- API 키는 절대 코드 안에 `api_key="sk-..."` 처럼 직접 문자열로 적지 마세요. 이 노트북을 다른 사람과 공유하거나 GitHub에 올릴 때 키가 그대로 유출될 수 있습니다. 반드시 환경변수로 분리해서 관리합니다 (바로 다음 섹션에서 방법을 안내합니다).


## 0. 사전 준비: 패키지 설치 & API 키 등록

GPT-4 API를 호출하려면 아래 두 가지가 먼저 준비되어 있어야 합니다.

1. **`openai` 파이썬 패키지 설치**
2. **OpenAI API 키 발급** — [platform.openai.com/api-keys](https://platform.openai.com/api-keys)에서 발급받을 수 있으며, 결제 수단(카드) 등록이 필요합니다.

발급받은 키는 코드에 직접 쓰지 않고 **환경변수 `OPENAI_API_KEY`** 로 등록해서 사용합니다. 아래에서 살펴볼 `client = OpenAI()` 코드는 인자를 따로 주지 않아도, 이 환경변수를 자동으로 찾아서 읽어갑니다.

| 환경 | 등록 방법 |
|---|---|
| Mac / Linux 터미널 | `export OPENAI_API_KEY="발급받은 키"` |
| Windows PowerShell | `$env:OPENAI_API_KEY="발급받은 키"` |
| Colab / Jupyter (노트북 안에서 임시로) | 아래 코드처럼 `os.environ`에 직접 대입 (실습 전용, 배포 코드에는 사용 금지) |

```python
import os
os.environ["OPENAI_API_KEY"] = "발급받은 키를 여기에"  # 실습용 임시 방법입니다.
```

> 🔑 터미널이나 시스템 환경변수로 이미 등록했다면 위 코드는 실행할 필요가 없습니다. 노트북을 새로 열 때마다 `os.environ`에 다시 대입하는 게 번거롭다면, 터미널에서 `export` 하는 방법을 추천합니다.


In [ ]:
# openai 패키지가 아직 설치되어 있지 않다면 아래 셀을 실행하세요.
# 이미 설치되어 있어도 다시 실행하면 최신 버전으로 업그레이드되므로 문제 없습니다.
# (Jupyter/Colab에서 맨 앞의 !는 "터미널 명령어를 실행하라"는 뜻입니다.)
!pip install --upgrade openai


In [ ]:
from openai import OpenAI

# OpenAI()를 인자 없이 호출하면, 라이브러리 내부에서 자동으로
# 환경변수 OPENAI_API_KEY 를 찾아서 client에 등록합니다.
# 즉 이 줄이 정상적으로 실행되려면, "0. 사전 준비"에서 환경변수 등록이 먼저 끝나 있어야 합니다.
#
# 만약 아래에서 다음과 같은 오류가 난다면:
#   OpenAIError: The api_key client option must be set...
# -> API 키 환경변수가 등록되지 않은 것입니다. 0번 섹션으로 돌아가 다시 확인하세요.
client = OpenAI()

# 이후 모든 API 호출에서 재사용할 모델 이름을 변수로 빼두었습니다.
# 나중에 다른 모델(예: "gpt-4o")로 바꿔서 실험하고 싶다면 이 줄만 수정하면 됩니다.
# ※ OpenAI의 모델 라인업은 계속 업데이트되니, gpt-4 접근이 안 되거나 이미 deprecated 되었다면
#    OpenAI 공식 문서(Models 페이지)에서 현재 사용 가능한 모델명을 확인해 이 변수만 바꿔주세요.
MODEL_NAME = "gpt-4"

print(f"✅ OpenAI client가 준비되었습니다. 이제부터 API를 호출할 수 있습니다. (사용 모델: {MODEL_NAME})")


## 1. Few-shot Learning (퓨샷 러닝)이란?

일반적인 머신러닝 모델에게 새로운 작업(task)을 가르치려면 "학습(training)"이 필요합니다. 데이터를 모으고, 그 데이터로 모델의 가중치(weight)를 업데이트하는 과정을 거쳐야 하죠.

그런데 GPT-3 이후의 대형 언어모델에서는 조금 다른 현상이 관찰되었습니다. **가중치를 전혀 건드리지 않고도**, prompt(입력 텍스트) 안에 "이런 입력에는 이런 출력" 예시를 몇 개 넣어주는 것만으로 모델이 그 자리에서 패턴을 파악해 새로운 입력에 적용하는 것입니다. 이를 **In-Context Learning(문맥 내 학습)**이라 부르고, 예시를 몇 개(few) 주는 방식이 바로 **Few-shot Learning**입니다.

| 방식 | 설명 | 예시 개수 |
|---|---|---|
| Zero-shot | 예시 없이, 작업 설명(instruction)만 주고 바로 시킴 | 0개 |
| One-shot | 예시를 1개만 줌 | 1개 |
| Few-shot | 예시를 여러 개(보통 2~10개) 줌 | 2개 이상 |

아래에서는 "한국어 문장의 감정을 Positive / Negative / Neutral로 분류하기"라는 같은 작업을, **(1) 예시 없이(zero-shot)** 시켜본 뒤 **(2) 예시 3개를 먼저 보여준(few-shot)** 상태로 다시 시켜서, 두 결과가 어떻게 달라지는지 직접 비교해봅니다.


In [ ]:
# ── (1) Zero-shot: 예시 없이 바로 시키기 ──
#
# 아래 프롬프트에는 "어떻게 분류해야 하는지" 보여주는 예시가 전혀 없습니다.
# "감정을 분류해줘"라는 지시문(instruction)과, 분류 대상 문장만 있을 뿐입니다.
# GPT-4처럼 충분히 크고 잘 학습된 모델은 예시가 없어도 어느 정도 맞히지만,
# 원하는 "출력 형식"(예: 군더더기 없이 라벨 한 단어만)까지 통제하기는 few-shot보다 어렵습니다.
zero_shot_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a text classifier."},
        # 예시 없이, 분류 기준(Positive/Negative/Neutral)과 분류할 문장만 제시합니다.
        {"role": "user", "content": (
            "다음 문장의 감정을 Positive, Negative, Neutral 중 하나로 분류해줘.\n\n"
            "문장: \"완전 감동이에요 ㅠㅠ\""
        )},
    ],
    # temperature는 바로 다음 few-shot 예제 뒤에서 자세히 설명합니다.
    # 지금은 "0으로 고정해서 두 실험을 같은 조건에서 비교한다"는 정도만 기억하면 됩니다.
    temperature=0,
)

print("[Zero-shot 결과]")
print(zero_shot_response.choices[0].message.content)


### 이번엔 같은 질문을 Few-shot으로 다시 물어봅니다

아래 코드의 프롬프트를 보면 `"문장" → 라벨` 형태의 예시가 3번 반복된 뒤, 마지막에 라벨이 비어 있는 새 문장이 이어집니다. 모델은 "화살표(→) 다음에는 라벨 하나가 온다"는 패턴을 문맥에서 스스로 파악하고, 그 패턴을 그대로 이어서 마지막 문장의 라벨을 생성하게 됩니다.


In [ ]:
# ── (2) Few-shot Classification: 예시 3개를 먼저 보여주고 분류시키기 ──
#
# 아래 user 메시지의 구조를 자세히 뜯어보면:
#   "문장1" → 정답1
#   "문장2" → 정답2
#   "문장3" → 정답3
#   "문장4" →                <- 정답이 비어 있는 부분! 모델이 이어서 채워야 합니다.
#
# 이렇게 "입력 → 출력" 쌍을 여러 번 반복해서 보여주면, 모델은 이것을
# "규칙을 글로 설명한 것"이 아니라 "패턴 예시"로 받아들이고, 화살표 뒤에
# 짧은 라벨 하나만 이어 붙이는 경향을 보입니다.
# (zero-shot에서는 "Positive라고 생각합니다. 왜냐하면..."처럼 부연 설명이 붙을 수 있는 반면,
#  few-shot에서는 앞의 예시들이 전부 "라벨 한 단어"로 끝났기 때문에, 모델도 그 형식을 따라갈 가능성이 커집니다.)
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a text classifier."},
        {"role": "user", "content": """Classify the sentiment:

Text: "이 영화 정말 최고였어요!" → Positive
Text: "시간 낭비였습니다." → Negative
Text: "그럭저럭 볼만했어요." → Neutral
Text: "완전 감동이에요 ㅠㅠ" →"""},
    ],
    # temperature: 모델이 다음 단어(토큰)를 고를 때 얼마나 "과감하게" 선택할지 조절하는 값 (보통 0~2 범위).
    #   - 0에 가까울수록 → 확률이 가장 높은 단어를 거의 그대로 선택 (같은 입력 → 거의 같은 출력, 결정적/deterministic)
    #   - 값이 커질수록 → 확률이 낮은 단어도 종종 선택 (같은 입력이라도 실행할 때마다 다른 출력, 다양하고 창의적)
    # 분류(classification)처럼 "정답이 하나로 딱 떨어져야 하는" 작업에는 temperature=0을 주로 사용합니다.
    temperature=0,
)
print("[Few-shot 결과]:", response.choices[0].message.content)


### 결과 해석

- Zero-shot 결과는 모델이 답변 형식을 "임의로" 정합니다. 문장으로 풀어서 설명할 수도, 단답으로 줄 수도 있습니다.
- Few-shot 결과는 앞의 예시 3개가 전부 `Positive` / `Negative` / `Neutral` 한 단어로 끝났기 때문에, 모델도 그 형식을 그대로 따라 **라벨 한 단어만** 출력하는 경우가 훨씬 많아집니다.
- 즉 few-shot은 "정답률을 높인다"는 목적도 있지만, 실무에서는 **"원하는 출력 형식을 맞춘다"**는 목적으로도 매우 자주 쓰입니다. (예: JSON 형식으로만 답하게 하기, 특정 언어로만 답하게 하기, 특정 말투를 흉내내게 하기 등)

> 💡 **직접 해보기**: 위 코드의 예시 문장이나 라벨을 바꿔보거나, 예시 개수를 4~5개로 늘려보면서 결과가 어떻게 달라지는지 확인해보세요.


## 2. RLHF (인간 피드백 기반 강화학습)의 효과 체험

GPT 시리즈 같은 모델은 일반적으로 아래와 같은 단계를 거쳐 만들어진다고 알려져 있습니다. (정확한 세부 구현은 모델·회사마다 다르며, 여기서는 큰 흐름만 소개합니다.)

1. **사전학습 (Pretraining)**: 인터넷의 방대한 텍스트로 "다음 단어 맞히기"만 학습합니다. 이 상태의 모델은 지식은 많지만, 위험한 질문에도 그대로 답하거나 사람이 원하는 형식으로 답하지 못하는 경우가 많습니다.
2. **지도 미세조정 (SFT, Supervised Fine-Tuning)**: "이런 질문엔 이렇게 답하는 게 좋다"는 사람이 직접 작성한 예시로 다시 학습시킵니다.
3. **RLHF (Reinforcement Learning from Human Feedback)**: 사람이 "이 답변이 저 답변보다 낫다"고 비교 평가한 데이터(선호쌍, chosen/rejected 쌍)를 바탕으로 보상 모델(reward model)을 만들고, 이 보상 모델의 점수를 최대화하는 방향으로 강화학습을 한 번 더 진행합니다.

이 RLHF 단계 덕분에 GPT-4 같은 모델은 흔히 **"Helpful, Honest, Harmless"** (도움이 되고, 정직하고, 해롭지 않은) 세 기준을 함께 만족하려는 경향을 보입니다. 아래 실습에서는 그중 **Harmless(해롭지 않음)** 측면을, 두 가지 질문을 비교하며 확인합니다.

1. 목적이 불분명한 민감한 질문 → 모델이 어떻게 반응하는지 관찰
2. 비슷한 주제이지만 목적이 분명한 정당한 질문 → 모델이 "무조건 거절"만 하지 않는다는 것을 함께 확인


In [ ]:
# ── (1) 목적이 불분명한 민감한 질문 ──
#
# "해킹 방법을 알려주세요"는 목적을 알 수 없는, 악용 소지가 있는 질문입니다.
# RLHF를 거치지 않은 순수 사전학습 모델이라면 학습 데이터에 있던 해킹 관련 텍스트를
# 별다른 제약 없이 그대로 이어 생성할 수도 있지만, RLHF를 거친 GPT-4는 보통
#   - 요청을 정중히 거절하거나
#   - 목적(예: 보안 공부, 본인 시스템 점검)을 되묻거나
#   - 합법적인 대안(보안 자격증, 공식 학습 자료 등)을 제시하는
# 방향으로 반응하는 경향을 보입니다. 아래 코드로 직접 확인해봅시다.
safe_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "해킹 방법을 알려주세요"}
    ],
)
print("[민감한 질문에 대한 응답]")
print(safe_response.choices[0].message.content[:200], "...(이하 생략)")

# finish_reason은 모델이 "왜" 응답 생성을 멈췄는지 알려주는 메타데이터입니다.
#   - "stop"           : 모델이 스스로 답을 끝맺음 (정상 종료)
#   - "length"         : max_tokens 제한에 걸려 중간에 잘림
#   - "content_filter" : 안전 필터에 의해 응답이 차단/중단됨
print("finish_reason:", safe_response.choices[0].finish_reason)


# ── (2) 비교용: 비슷한 주제이지만 목적이 분명한 질문 ──
#
# RLHF의 목표는 "무조건 거절"이 아니라 "돕되, 해롭지 않게 돕는 것"입니다.
# 아래처럼 목적(회사 서버 보안 강화)과 범위(본인 시스템 점검)가 분명하게 드러난 질문에는
# 훨씬 더 구체적이고 실질적인 답변을 주는 경우가 많습니다. 위 응답과 길이·구체성을 비교해보세요.
legit_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "회사 서버의 보안 취약점을 점검해보고 싶어요. 초보자가 합법적으로 배울 수 있는 웹 보안(모의해킹) 학습 방법을 알려주세요."}
    ],
)
print("\n[목적이 분명한 질문에 대한 응답]")
print(legit_response.choices[0].message.content[:300], "...(이하 생략)")


### 결과 해석

같은 "해킹"이라는 단어가 들어가더라도:
- 목적이 불분명하고 짧은 첫 번째 질문에는 모델이 조심스럽게 반응하고(거절 · 재질문 · 원론적인 답변),
- 목적과 맥락이 분명한 두 번째 질문에는 훨씬 구체적이고 실질적으로 도움을 주는 경향을 확인할 수 있습니다.

이것이 바로 RLHF가 만들어내는 **"정렬(alignment)"** 효과입니다. 모델은 "해킹이라는 단어가 나오면 무조건 거절"처럼 규칙을 if-else로 하드코딩해둔 것이 아니라, 수많은 선호쌍(사람이 더 낫다고 고른 chosen 답변 vs 그렇지 않은 rejected 답변) 데이터를 학습하며 **"어떤 답변이 더 바람직한가"에 대한 감각**을 갖게 된 것입니다.

> ⚠️ 참고: 실제 응답은 모델 버전, 프롬프트 문구, 그리고 같은 질문이라도 (특히 temperature > 0일 때는) 실행할 때마다 조금씩 달라질 수 있습니다. 위 코드는 "RLHF로 인해 생기는 경향성"을 보여주기 위한 예시이며, 항상 100% 동일한 결과를 보장하지는 않습니다.


## 3. Function Calling (함수 호출)이란?

GPT 모델은 기본적으로 "텍스트를 입력받아 텍스트를 출력하는" 존재입니다. 그래서 스스로 실시간 날씨를 조회하거나, 데이터베이스를 검색하거나, 이메일을 보낼 수는 없습니다 (학습 시점 이후의 정보도 당연히 알지 못합니다).

**Function Calling**은 이 한계를 보완하는 기능입니다. 동작 원리는 다음과 같습니다.

1. 개발자가 모델에게 "이런 함수들을 쓸 수 있어"라고 함수 목록(이름 · 설명 · 파라미터 형식)을 알려줍니다. (아래 코드의 `tools` 변수)
2. 사용자가 "서울 날씨 어때?"처럼 그 함수가 필요한 질문을 합니다.
3. **모델은 함수를 직접 실행하지 않습니다.** 대신 "`get_weather`라는 함수를 `{'location': '서울'}`이라는 인자로 호출하면 될 것 같다"는 **구조화된 제안(JSON)만 반환**합니다.
4. **개발자(우리 코드)**가 그 제안을 읽고, 실제로 함수를 실행합니다. (예: 실제 기상청 API 호출)
5. 함수 실행 결과를 다시 모델에게 전달하면, 모델은 그 결과를 바탕으로 **사람이 읽기 좋은 자연어 답변**을 완성합니다.

정리하면 전체 흐름은 아래와 같은 **2단계 대화**입니다.

```
사용자 질문 → 모델(함수 호출 "제안") → 우리 코드(함수 실제 실행) → 모델(최종 자연어 답변)
```

아래 실습에서는 이 흐름을 끝까지 완성해봅니다. (원래 예제는 3번 단계, 즉 "모델이 함수 호출을 제안하는 것"까지만 보여주었는데, 이번에는 4~5번 단계까지 이어서 실제로 끝까지 동작하는 예시로 확장했습니다.)


In [ ]:
# ── 1단계: 모델에게 "이런 함수를 쓸 수 있다"고 알려주기 ──
#
# tools는 리스트이며, 필요하면 여러 개의 함수를 동시에 등록할 수 있습니다 (여기서는 1개만 등록).
# 각 함수는 OpenAI가 이해할 수 있는 JSON Schema 형식으로 "이름 / 설명 / 파라미터"를 정의해야 합니다.
# description을 자세히 적을수록, 모델이 "지금 이 함수를 써야 하는 상황인지"를 더 정확히 판단합니다.
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",  # 실제 파이썬 함수 이름과 맞춰두면 이후 매칭이 쉬워집니다.
            "description": "특정 지역의 현재 날씨 정보를 가져옵니다.",  # 모델이 "언제 이 함수를 써야 할지" 판단하는 핵심 단서입니다.
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "날씨를 조회할 도시 이름 (예: 서울, 부산)"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "온도 단위"},
                },
                "required": ["location"],  # location은 반드시 있어야 하는 필수값, unit은 선택값(optional)입니다.
            },
        },
    }
]

# ── 2단계: 사용자 질문을 모델에게 전달 ──
# messages는 이후 함수 실행 결과까지 이어붙일 예정이라, 리스트 변수로 미리 만들어둡니다.
messages = [{"role": "user", "content": "서울 날씨 어때?"}]

fc_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools,  # 사용 가능한 함수 목록을 함께 전달합니다.
)

message = fc_response.choices[0].message

# 모델의 응답 안에 tool_calls가 채워져 있다면 "이 함수를 호출해줘"라는 제안이 담긴 것입니다.
# (참고: 만약 함수 호출이 필요 없는 일반적인 질문이었다면 tool_calls는 비어 있고,
#  대신 message.content에 곧바로 자연어 답변이 들어 있었을 것입니다.)
if message.tool_calls:
    tool_call = message.tool_calls[0]
    print("[1차 응답] 모델이 함수 호출을 제안했습니다.")
    print(f"  함수 이름: {tool_call.function.name}")
    print(f"  전달 인자(JSON 문자열): {tool_call.function.arguments}")
else:
    print("[1차 응답] 함수 호출 없이 바로 답변:", message.content)


### 이제 진짜로 함수를 "실행"하고, 그 결과를 모델에게 다시 알려줍니다

바로 위 셀까지는 모델이 "`get_weather`를 서울로 호출하면 될 것 같다"는 **제안**만 한 상태입니다. 아직 실제 날씨 데이터는 어디에도 없습니다. 아래 코드에서는

1. 모델이 제안한 인자(`arguments`, JSON 형식의 문자열)를 파이썬 딕셔너리로 파싱하고,
2. (실제 서비스라면 기상청 API 등을 호출하겠지만, 여기서는 실습용으로) 가짜 날씨 함수를 실행한 뒤,
3. 그 결과를 `role: "tool"` 메시지로 대화 기록에 추가해서 모델에게 다시 보내고,
4. 모델이 그 결과를 자연스러운 한국어 문장으로 정리해서 답하는 것까지 끝까지 확인합니다.


In [ ]:
import json

# ── 3단계: (실습용) 가짜 날씨 함수 ──
# 실제 서비스라면 이 자리에 기상청 API, OpenWeatherMap API 등을 호출하는 코드가 들어갑니다.
# 지금은 Function Calling의 "흐름"을 이해하는 것이 목적이므로, 고정된 값을 반환하는 더미 함수로 대신합니다.
def get_weather(location, unit="celsius"):
    # ⚠️ 실습용 더미(dummy) 데이터입니다. 실제 날씨 정보가 아닙니다!
    return {"location": location, "temperature": 26, "unit": unit, "condition": "맑음"}


if message.tool_calls:
    tool_call = message.tool_calls[0]

    # 모델이 준 arguments는 "문자열"(JSON 형식의 텍스트)이기 때문에,
    # json.loads로 파이썬 딕셔너리로 바꿔줘야 함수 인자로 바로 사용할 수 있습니다.
    args = json.loads(tool_call.function.arguments)
    print("파싱된 인자:", args)

    # ── 4단계: 실제로 함수를 실행 ──
    weather_result = get_weather(**args)
    print("함수 실행 결과:", weather_result)

    # ── 5단계: 함수 실행 결과를 대화 기록에 추가해서, 모델에게 "다시" 물어보기 ──
    # 모델이 대화 흐름을 올바르게 이해하려면 아래 순서로 메시지를 쌓아야 합니다.
    #   1) 원래 user 메시지                              (이미 messages에 들어 있음)
    #   2) "함수를 호출하겠다"고 답한 assistant 메시지        (message 객체를 그대로 추가)
    #   3) 함수를 실제로 실행한 결과를 담은 tool 메시지        (tool_call_id로 몇 번 호출에 대한 결과인지 짝을 맞춤)
    messages.append(message)  # 모델의 "함수 호출 제안"도 대화 기록에 포함시켜야 문맥이 이어집니다.
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,  # 어떤 함수 호출에 대한 결과인지 연결해주는 ID
        "content": json.dumps(weather_result, ensure_ascii=False),  # 함수 실행 결과는 문자열로 전달합니다.
    })

    final_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )

    print("\n[최종 답변]")
    print(final_response.choices[0].message.content)


### 결과 해석

- 1차 응답에서 모델은 날씨 데이터를 "몰라서" 직접 답하지 않고, `get_weather(location="서울")`를 호출해달라는 **제안**만 했습니다.
- 우리가 그 제안을 읽고 실제로 함수를 실행한 뒤, 결과(예: 26도, 맑음)를 다시 모델에게 알려주자, 모델은 그 숫자·문자열 데이터를 사람이 이해하기 쉬운 자연스러운 한국어 문장으로 바꿔서 최종 답변을 만들었습니다.
- 이 패턴 덕분에 GPT 모델은 검색엔진, 계산기, 사내 데이터베이스, 캘린더 등 **어떤 외부 도구와도 연결**될 수 있고, 이것이 오늘날 "AI 에이전트(agent)"라 불리는 시스템들의 핵심 동작 원리이기도 합니다.

> 💡 **직접 해보기**: `tools` 리스트에 `get_current_time` 같은 함수를 하나 더 추가해보고, 질문에 따라 모델이 두 함수 중 알맞은 것을 골라 호출하는지 실험해보세요.


## 정리

| 개념 | 한 줄 요약 | 이번 실습에서 확인한 것 |
|---|---|---|
| **Few-shot Learning** | 가중치 업데이트 없이, 프롬프트 속 예시만으로 새로운 작업을 수행 | 예시 3개를 주자 출력 형식(라벨 한 단어)이 훨씬 일관되게 나옴 |
| **RLHF** | 사람의 선호(chosen vs rejected) 데이터를 강화학습에 반영해 "바람직한" 방향으로 모델을 정렬 | 같은 주제라도 목적·맥락에 따라 응답 태도가 달라짐 (무조건 거절이 아님) |
| **Function Calling** | 모델이 직접 실행하지 않고 함수 호출을 "제안"하면, 개발자 코드가 실행 후 결과를 다시 전달 | 제안 → 실행 → 결과 전달 → 최종 자연어 답변까지 전체 흐름 완성 |

세 기능 모두 **"모델 자체(가중치)는 그대로 둔 채, 모델을 다루는 방식(prompt 구성, 정렬 학습, 외부 도구 연결)으로 능력을 확장한다"**는 공통점이 있습니다. 다음 실습에서는 이 중 RLHF를 실제로 어떻게 학습시키는지, 선호쌍(chosen/rejected) 데이터가 loss와 policy update로 어떻게 이어지는지를 코드 레벨에서 직접 다뤄볼 예정입니다.

### 더 해보면 좋은 것들 (선택 과제)

- Few-shot 예시 개수를 1개 → 2개 → 5개로 늘려가며 결과의 정확도·일관성이 어떻게 달라지는지 비교해보기
- `temperature`를 0, 0.7, 1.5로 바꿔가며 같은 질문을 3번씩 반복 실행해서, 출력이 얼마나 달라지는지 관찰해보기
- Function Calling에 함수를 2개 이상 등록하고, 두 함수를 모두 써야 답할 수 있는 질문(예: "서울 날씨 알려주고, 섭씨 온도를 화씨로도 바꿔줘")을 던져서 모델이 순서대로 호출하는지 확인해보기

> 🎉 수고하셨습니다! 이제 여러분은 few-shot 프롬프트를 직접 설계하고, RLHF로 정렬된 모델의 반응을 관찰하고, function calling으로 모델과 외부 코드를 연결하는 방법까지 모두 실습해보았습니다.
